## Setting Up Enviroment

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

Matplotlib is building the font cache; this may take a moment.


## Loading The Data 

In [ ]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'var_defs': {
        'local': '../data/raw/VariableDefinitions.csv',
        # Fixed typo: added 's' to VariableDefinitions.csv
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/VariableDefinitions.csv'
    }
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
var_defs = data['var_defs']


Loaded train from local path.
Loaded test from local path.
Loaded var_defs from local path.


In [49]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Variable definitions", var_defs)

Train shape: (18506, 21)
Test shape: (6169, 20)
Variable definitions               Column Name                                         Definition
0                      id                 Unique identifier for each tourist
1                 country                The country a tourist coming  from.
2               age_group                        The age group of a tourist.
3             travel_with  The relation of people a tourist travel with t...
4            total_female                            Total number of females
5              total_male                              Total number of males
6                 purpose                  The purpose of visiting  Tanzania
7           main_activity           The main activity of tourism in Tanzania
8            infor_source  The source of information about tourism in Tan...
9         tour_arrangment                The arrangment of visiting Tanzania
10  package_transport_int  If the tour package include international tran...
11   pa

## Structural Overview

In [50]:
# Structural Overview for the Training Data
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 18506 entries, 0 to 18505
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18506 non-null  str    
 1   country                18506 non-null  str    
 2   age_group              18506 non-null  str    
 3   travel_with            17431 non-null  str    
 4   total_female           18504 non-null  float64
 5   total_male             18500 non-null  float64
 6   purpose                18506 non-null  str    
 7   main_activity          18506 non-null  str    
 8   info_source            18506 non-null  str    
 9   tour_arrangement       18506 non-null  str    
 10  package_transport_int  18506 non-null  str    
 11  package_accomodation   18506 non-null  str    
 12  package_food           18506 non-null  str    
 13  package_transport_tz   18506 non-null  str    
 14  package_sightseeing    18506 non-null  str    
 15  package_guide

In [51]:
train.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,Yes,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,No,6,6,Yes,Higher Cost
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,3,0,No,Lower Cost
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,No,Yes,Yes,7,0,Yes,Higher Cost


In [41]:
train.tail()

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
18501,tour_idmp5ciw50,KENYA,45-64,Alone,0.0,1.0,Other,Hunting Tourism,Others,Independent,No,No,No,No,No,No,No,3,0,No,Lower Cost
18502,tour_ideq0yncfs,KENYA,45-64,Alone,1.0,0.0,Meetings and Conference,Wildlife Tourism,Others,Independent,No,No,No,No,No,No,No,2,0,No,Lower Cost
18503,tour_idv7pz3vs8,UNITED STATES OF AMERICA,25-44,With Spouse and Children,2.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Independent,No,No,No,No,No,No,No,9,0,Yes,Higher Cost
18504,tour_idy6ydo00w,UNITED STATES OF AMERICA,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,No,13,4,No,Higher Cost
18505,tour_idceoq9por,OMAN,25-44,With Spouse and Children,2.0,1.0,Visiting Friends and Relatives,Wildlife Tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,22,0,No,Low Cost


In [33]:
train.dtypes.value_counts()

str        17
float64     2
int64       2
Name: count, dtype: int64

In [ ]:
# Training Data Descriptive Statistics
train.describe()

,total_female,total_male,night_mainland,night_zanzibar
count,18504.000000,18500.000000,18506.000000,18506.000000
mean,0.936230,0.998757,9.141576,2.493516
std,1.215582,1.173177,14.127449,5.275156
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,3.000000,0.000000
50%,1.000000,1.000000,6.000000,0.000000
75%,1.000000,1.000000,11.000000,4.000000
max,49.000000,58.000000,365.000000,240.000000


In [35]:
# Structural Overview for the Testing Data
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 6169 entries, 0 to 6168
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                6169 non-null   str    
 1   country                6169 non-null   str    
 2   age_group              6169 non-null   str    
 3   travel_with            5808 non-null   str    
 4   total_female           6167 non-null   float64
 5   total_male             6168 non-null   float64
 6   purpose                6169 non-null   str    
 7   main_activity          6169 non-null   str    
 8   info_source            6169 non-null   str    
 9   tour_arrangement       6169 non-null   str    
 10  package_transport_int  6169 non-null   str    
 11  package_accomodation   6169 non-null   str    
 12  package_food           6169 non-null   str    
 13  package_transport_tz   6169 non-null   str    
 14  package_sightseeing    6169 non-null   str    
 15  package_guided_

In [37]:
test.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
0,tour_idynufedne,KOREA,25-44,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,Others,Independent,No,No,No,No,No,No,No,7,4,Yes
1,tour_id9r3y5moe,UNITED KINGDOM,45-64,With Children,1.0,1.0,Leisure and Holidays,Conference Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,Yes,7,0,Yes
2,tour_idf6itml6g,ITALY,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,No,Yes,No,No,No,0,6,Yes
3,tour_id99u4znru,KENYA,25-44,Alone,0.0,1.0,Other,Beach Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,3,4,No
4,tour_idj4i9urbx,ZAMBIA,25-44,Alone,0.0,1.0,Business,Widlife Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,6,0,No


In [38]:
test.tail()

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
6164,tour_id2deyfjhq,ZIMBABWE,25-44,Alone,0.0,1.0,Business,Wildlife Tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,2,0,No
6165,tour_idlenv2rio,DRC,25-44,Alone,0.0,1.0,Visiting Friends and Relatives,Hunting Tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,60,0,Yes
6166,tour_id7wwqrs0p,CANADA,25-44,Alone,0.0,1.0,Leisure and Holidays,Beach Tourism,"Friends, relatives",Package Tour,Yes,Yes,Yes,Yes,No,No,No,5,0,No
6167,tour_idx80vbw5a,CANADA,18-24,Alone,1.0,0.0,Visiting Friends and Relatives,Wildlife Tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,21,0,No
6168,tour_id8fkkwytb,KENYA,45-64,NaN,0.0,1.0,Meetings and Conference,Wildlife Tourism,"Friends, relatives",Independent,No,No,No,No,No,No,No,4,0,Yes


In [39]:
test.dtypes.value_counts()

str        16
float64     2
int64       2
Name: count, dtype: int64

In [ ]:
# Testing Data Descriptive Statistics
test.describe()

,total_female,total_male,night_mainland,night_zanzibar
count,6167.000000,6168.000000,6169.00000,6169.000000
mean,0.922491,1.017510,9.31172,2.585832
std,1.173067,1.526274,16.36690,5.465058
min,0.000000,0.000000,0.00000,0.000000
25%,0.000000,1.000000,3.00000,0.000000
50%,1.000000,1.000000,6.00000,0.000000
75%,1.000000,1.000000,11.00000,4.000000
max,30.000000,90.000000,664.00000,174.000000


## Checking for missing values

In [11]:
# Checking for missing training data
missing_train = train.isnull().sum().sort_values(ascending=False)
missing_train_pct = (missing_train / len(train) * 100).round(2)
missing_summary = pd.concat([missing_train, missing_train_pct], axis=1, keys=['count', 'pct'])
missing_summary[missing_summary['count'] > 0]

,count,pct
travel_with,1075,5.81
total_male,6,0.03
total_female,2,0.01


In [12]:
# Checking for missing testing data
missing_test = test.isnull().sum().sort_values(ascending=False)
missing_test_pct = (missing_test / len(test) * 100).round(2)
missing_summary = pd.concat([missing_test, missing_test_pct], axis=1, keys=['count', 'pct'])
missing_summary[missing_summary['count'] > 0]

,count,pct
travel_with,361,5.85
total_female,2,0.03
total_male,1,0.02


In [44]:
# Checking for duplicates in the training data
duplicated_rows_in_training_data = train.duplicated().sum()
print(f"Number of duplicated rows in the training data: {duplicated_rows_in_training_data}")

Number of duplicated rows in the training data: 0


In [45]:
# Checking for duplicates in the testing ata
duplicated_rows_in_testing_data = test.duplicated().sum()
print(f"Number of duplicated rows in the testing data: {duplicated_rows_in_testing_data}")

Number of duplicated rows in the testing data: 0
